### Group info

ID: G5_4

Members:
- Vinicius Millan Santa Catharina
- Zaur Kambiev
- Stig Espen Paulsberg

### Imports

In [1]:
import numpy as np
import pandas as pd
from numpy.typing import ArrayLike, NDArray

### Task 1

### Task 2

In [ ]:
from pathlib import Path

def directory_search(folder_name: str | Path) -> list[Path]:
    """Search a directory and its subdirectories for .log files.

    Args:
        folder_name: The path to the directory to search.

    Returns:
        A list of Path objects representing the matching log files.
    """
    files = []

    # Convert the input to a pathlib.Path object
    directory = Path(folder_name) 

    # Iterate over the items (file or directory) in the current directory
    for item in directory.iterdir():

        # Add matching log files to the results
        if item.is_file() and item.suffix == '.log':
            files.append(item)
            
        elif item.is_dir():
            # Recursively search the subdirectory
            files.extend(directory_search(item))

    return files               


folder = '../data'
print(directory_search(folder))

In [ ]:
import json

with open(r'../data/store_config.json', 'r') as file:
    content = json.load(file)


def inspect_json(content: dict | list) -> set[str]:
    """Recursively inspect JSON content and return active register ids.
    
        Args:
            content: A dictionary or list containing JSON data.
    
        Returns:
            A set of ids for registers whose active status is True.
        """
    ids = set()

    # Check whether the current content is a dictionary
    if type(content) is dict:
        for key, value in content.items():

            # Add the register id if the register is active
            if key == 'id' and content.get('active') is True:
                ids.add(content['id'])

            # Recursively inspect nested dictionaries and lists
            if type(value) is dict or type(value) is list:
                ids.update(inspect_json(value))

    # Check whether the current content is a list
    elif type(content) is list:
        for item in content:
            ids.update(inspect_json(item))
    
    return ids       


print(inspect_json(content))


### Task 3

Define a function for handling price outliers.

In [2]:
def handle_outliers(prices: ArrayLike) -> NDArray[np.float64]:
    """Replace price outliers with the median of valid prices.

    Outliers are defined as prices more than three standard deviations
    away from the mean.

    Parameters
    ----------
    prices : array_like
        Prices convertible to a 1D NumPy array of floats.

    Returns
    -------
    ndarray
        A 1D NumPy array of prices with outliers replaced by the
        median of valid prices.
    """

    prices = np.asarray(prices, dtype=np.float64)

    if prices.ndim != 1:
        raise ValueError("Prices must be 1D.")

    mean = prices.mean()
    std = prices.std()

    valid_mask = abs(prices - mean) <= 3 * std

    median = np.median(prices[valid_mask])

    prices[~valid_mask] = median

    return prices

Define a function for calculating the refund ratio of items.

In [3]:
EPS = 1e-6


def compute_refund_ratio(
        sales_count: NDArray,
        refund_count: NDArray
) -> NDArray:
    """Compute the refund-to-sale ratio for each item.

    A small positive constant (EPS) is added to the number of sales
    to prevent division by zero.

    Parameters
    ----------
    sales_count : ndarray
        Number of sales for each item.
    refund_count : ndarray
        Number of refunds for each item.

    Returns
    -------
    ndarray
        An array containing element-wise refund-to-sale ratios.
    """

    return refund_count / (sales_count + EPS)

### Task 4

Define a function that converts a list of transactions into a `pandas.DataFrame`.

In [4]:
def list_to_df(transactions: list[dict]) -> pd.DataFrame:
    """Convert a list of transactions into a pandas DataFrame.

    Convert the timestamp column to datetime, set it as the DataFrame index,
    and sort the transactions chronologically.

    Parameters
    ----------
    transactions : list[dict]
        A list of dictionaries containing a `timestamp` field
        in YYYY-MM-DD HH:MM:SS format.

    Returns
    -------
    pd.DataFrame
        A DataFrame indexed by timestamp, sorted in chronological order.
    """

    transactions = pd.DataFrame(transactions)

    transactions["timestamp"] = pd.to_datetime(transactions["timestamp"])

    transactions = transactions.set_index("timestamp").sort_index()

    return transactions

Define a function that resamples transactions into 1-hour intervals. 

In [5]:
def resample_hourly(transactions: pd.DataFrame) -> pd.DataFrame:
    """Aggregate transaction prices into hourly intervals by item.

    We group transactions only by item and ignore all differences
    in other columns (e.g. currency, status).

    Parameters
    ----------
    transactions : pd.DataFrame
        Transaction records with a DatetimeIndex and columns for item, price.

    Returns
    -------
    pd.DataFrame
        Hourly price totals grouped by item.
    """

    return (
        transactions
        .groupby("item")
        .resample("h")
        .agg(total_price=("price", "sum"))
    )

### Task 5